In [2]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bandini2021individually")
original_data_pathway = os.path.join(pathway, "original_data")

# complete_path_1 = os.path.join(original_data_pathway, "Bandini_emailed_2021_LPZ Orang nut-cracking data .csv")
complete_path_1 = os.path.join(original_data_pathway, "data nutcracking orangutans 160420.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)
df['study_id']="bandini2021individually"

df = df.rename(columns={"Subject": "ape"})
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df.columns

In [5]:
# df['date of session']= pd.to_datetime(df['date of session'],format='%m/%d/%Y')
df[['month','day', 'year']] = df['date of session'].str.split('/',expand=True)

df = df.rename(columns={"date of session": "date_original"})

df['trial'] = df['session number'] 
df = df.rename(columns={"session number": "session"})


In [6]:
# df.columns

In [7]:
df = df.rename(columns={"type of nut": "type_of_nut",
    "tape number": "tape_number",
    "begin on tape": "begin_on_tape",
    "took at least one nut y/n": "took_at_least_one_nut_yes_or_no",
    "hammering on nut y/n": "hammering_on_nut_yes_or_no",
    "cracked nuts by hammering y/n": "cracked_nuts_by_hammering_yes_or_no",
    "# of nuts cracked open by hammering": "number_of_nuts_cracked_open_by_hammering",
    "cracked nut with mouth y/n": "cracked_nut_with_mouth_yes_or_no",
    "# of nuts cracked open with mouth": "number_of_nuts_cracked_open_with_mouth",
    "placed nut on anvil?": "placed_nut_on_anvil",
    "put nut in hole on anvil?": "put_nut_in_hole_on_anvil",
    "anvil involved in cracking y/n": "anvil_involved_in_cracking_yes_or_no",
    "other crack methods ??": "other_crack_methods"})

In [8]:
df['ape'].replace('', np.nan, inplace=True)
df.dropna(subset=['ape'], inplace=True)
df.rename(columns={"ape": "participant"}, inplace=True)
df['condition'].replace(' ', '_', inplace=True,regex=True)
df['comments'].replace(', ', '-', inplace=True,regex=True)

replace_list = ['hammering_on_nut_yes_or_no', 'put_nut_in_hole_on_anvil', 'other_crack_methods','comments']
for x in replace_list:
    df[x].replace(' ', '_', inplace=True,regex=True)

In [9]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [10]:
bandini2021individually_standardized=df[['study_id', 'year', 'month', 'day',
       'participant','age_in_years',  'sex', 'species', 
       'session',
        #  'trial',
       'condition', 'type_of_nut',
        #  'tape_number',
       'begin_on_tape',  'took_at_least_one_nut_yes_or_no',
       'hammering_on_nut_yes_or_no', 'cracked_nuts_by_hammering_yes_or_no',
       'number_of_nuts_cracked_open_by_hammering', 'cracked_nut_with_mouth_yes_or_no',
       'number_of_nuts_cracked_open_with_mouth', 'placed_nut_on_anvil',
       'put_nut_in_hole_on_anvil', 'anvil_involved_in_cracking_yes_or_no',
       'other_crack_methods', 'comments']]

In [11]:
comp_out_path_stand = os.path.join(out_pathway, 'bandini2021individually_exp2_standardized.csv')
bandini2021individually_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [12]:
names =bandini2021individually_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
bandini2021individually_glossary=df[["column_name", "description"]]


comp_out_path_glossary = os.path.join(out_pathway, 'bandini2021individually_exp2_glossary.csv')
bandini2021individually_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
